# 01. Train

Mỗi lượt được lưu trên Drive trong `MyDrive/vihsd-lora/outputs/runs/<tên lượt>` và ghi lên W&B.

- **Checkpoint:** LoRA lưu mỗi epoch, full fine-tune lưu cách 2 epoch. Drive chỉ giữ checkpoint mới nhất.
- **Model tốt nhất:** lưu vào `best/` ngay khi macro F1 trên dev tăng.
- **Bị ngắt phiên:** chạy lại toàn bộ notebook với cùng lượt, quá trình tiếp tục từ checkpoint gần nhất và nối vào cùng run trên W&B.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Repo private thì thêm secret GITHUB_TOKEN trên Colab.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "TEN-TAI-KHOAN" in REPO_URL:
        raise ValueError("Hãy sửa REPO_URL thành địa chỉ repo GitHub của bạn rồi chạy lại ô này.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab cài sẵn torchao bản cũ, peft sẽ báo lỗi khi gắn LoRA. Repo không dùng torchao nên gỡ đi.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Kết quả lưu tại:", settings.output_root)
print(f"Dung lượng còn trống: {free_gb} GB")
if free_gb < 6:
    print("Cảnh báo: còn dưới 6GB. Lượt full fine-tune cần khoảng 3GB, hãy dọn Drive và làm trống thùng rác.")

## Chọn lượt

| Lượt | Cách train | r | Epoch |
|---|---|---|---|
| A | LoRA | 8 | 10 |
| B | Full fine-tune | | 10 |
| C | LoRA | 2 | 10 |
| D | LoRA | 16 | 10 |
| E | LoRA | 8 | 30 |

Có thể chạy nhiều lượt liền nhau, hoặc chia mỗi thành viên một lượt nếu dùng chung một W&B project.

In [ ]:
RUNS_TO_TRAIN = ["A"]
SEED = 42
WANDB_TEAM = None  # Điền tên team W&B nếu cả nhóm dùng chung một không gian
SAVE_EVERY_EPOCHS = None  # Bỏ trống để dùng mặc định: LoRA 1 epoch, full fine-tune 2 epoch

settings.wandb_entity = WANDB_TEAM or settings.wandb_entity
settings.save_every_epochs = SAVE_EVERY_EPOCHS

In [ ]:
import gc
import pandas as pd
import torch
from vihsd_lora.train import train_run

summaries = []
for key in RUNS_TO_TRAIN:
    summaries.append(train_run(key, seed=SEED, settings=settings))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

columns = ["slug", "trainable_params", "trainable_pct", "dev_accuracy", "dev_macro_f1", "best_epoch",
           "mean_epoch_seconds", "gpu_peak_mb", "saved_mb"]
display(pd.DataFrame(summaries)[columns])

## Xem biểu đồ

Mở link W&B in ra trong lúc train. Biểu đồ theo epoch nằm ở nhóm `by_epoch`, gồm `dev_macro_f1`, `dev_accuracy`, `train_loss`, `epoch_seconds` và `gpu_peak_mb`. Chọn nhiều lượt trong bảng Runs để so trên cùng một biểu đồ.